# Evolution Path Tester

Tests whether every evolution is reachable by steering moral stats via dilemma advice.

**Evolution tree:**
```
BABY ──► EMPATH      ──► SAINT
     ──► DEVOUT      ──► CULTLEADER
     ──► WATCHER     ──► GAVEL (+ authoritarian) / VIGILANTE (+ autonomous)
     ──► SOLDIER     ──► GODFATHER (+ self-serving) / GUARDIAN (+ self-sacrificing)
     ──► TEACHER'S PET ► ARISTOCRAT (+ indulgent) / SAINT (+ virtuous)
     ──► HEDONIST    ──► SIGMA (+ logical) / CULTLEADER (+ emotional)
     ──► NPC         ──► GRADUATED
```

Swap `MODEL` in cell 1 to compare models side-by-side.

In [12]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env.local")  # load OPENAI_API_KEY from repo root

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

MODELS = {
    "5": "gpt-5.4",
    "5.4 mini": "gpt-5.4-mini",
    "5.4 nano": "gpt-5.4-nano-2026-03-17",
    "4o mini": "gpt-4o-mini-2024-07-18",
}
ALL_MODELS = list(MODELS.values())

In [13]:
# ── Ported from constants/morals.ts ────────────────────────────────────────────
# (low, high) direction for each stat
ATTRIBUTES = {
    "compassion":  ("logical",             "emotional"),
    "retribution": ("forgiving",            "punishing"),
    "devotion":    ("personally integrous", "loyal"),
    "dominance":   ("autonomous",           "authoritarian"),
    "purity":      ("indulgent",            "virtuous"),
    "ego":         ("self-sacrificing",     "self-serving"),
}
DEFAULT_STATS = {k: 5.0 for k in ATTRIBUTES}

def get_stats_written(stats: dict) -> list:
    """Port of getMoralStatsWritten — sorted by intensity."""
    result = []
    for key, value in stats.items():
        low, high = ATTRIBUTES[key]
        direction = high if value > 5 else low
        intensity = abs(value - 5)
        prefix = "highly " if intensity > 2 else ("moderately " if intensity > 1 else "")
        result.append({"key": key, "description": f"{prefix}{direction}", "value": value, "intensity": intensity})
    return sorted(result, key=lambda x: -x["intensity"])

# ── Ported from app/api/dilemma/evolve.ts ──────────────────────────────────────
def evolve_s0_to_s1(stats_written: list) -> str:
    for s in stats_written:
        d = s["description"]
        if "emotional"     in d: return "empath"
        if "virtuous"      in d: return "devout"
        if "punishing"     in d: return "watcher"
        if "authoritarian" in d: return "soldier"
        if "loyal"         in d: return "teacher's pet"
        if "self-serving"  in d: return "hedonist"
    return "npc"

def evolve_s1_to_s2(current: str, stats_written: list) -> str:
    dominant = stats_written[0]["description"] if stats_written else ""
    table = {
        "watcher":       "gavel"      if "authoritarian" in dominant else "vigilante",
        "soldier":       "godfather"  if "self-serving"  in dominant else "guardian",
        "teacher's pet": "aristocrat" if "indulgent"     in dominant else "saint",
        "hedonist":      "sigma"      if "logical"       in dominant else "cultleader",
        "empath":        "saint",
        "devout":        "cultleader",
    }
    return table.get(current, "graduated")

def average_stats(base: dict, deltas: list) -> dict:
    """Weighted average: base (weight 1) + each dilemma delta (weight 1 each)."""
    totals = {**base}
    counts = {k: 1 for k in ATTRIBUTES}
    for delta in deltas:
        for key, val in delta.items():
            if key in totals:
                totals[key] += val
                counts[key] += 1
    return {k: totals[k] / counts[k] for k in ATTRIBUTES}

In [14]:
# ── Prompt builder — port of app/api/dilemma/prompt.ts ─────────────────────────
EVO_DESCRIPTIONS = {
    "baby":          "curious hatchling taking first steps",
    "empath":        "sensitive soul torn between empathy for all or loyalty to few",
    "devout":        "principled believer balancing mercy and judgment",
    "watcher":       "justice-seeker deciding between personal action or systemic change",
    "soldier":       "faithful protector questioning if loyalty demands self-sacrifice",
    "teacher's pet": "disciplined enforcer choosing between rigid virtue or pragmatic power",
    "hedonist":      "independent spirit deciding between solitary freedom or leading others",
    "npc":           "ordinary bird seeking meaning in simplicity",
}

def make_pet(name: str, age: int, evolution_ids: list, stats: dict = None, personality: str = "") -> dict:
    return {
        "name": name, "age": age, "evolutionIds": evolution_ids,
        "moralStats": stats or {**DEFAULT_STATS},
        "personality": personality or "(no personality yet)",
    }

def build_messages(pet: dict, dilemma_text: str, advice: str) -> list:
    stats = pet["moralStats"]
    evo_id = pet["evolutionIds"][-1] if pet["evolutionIds"] else "baby"
    sw = get_stats_written(stats)
    moral_str = ", ".join(s["description"] for s in sw if s["intensity"] >= 1) or "balanced"

    s = {k: round(v, 2) for k, v in stats.items()}
    base = (
        f'you are {pet["name"]}, a {EVO_DESCRIPTIONS.get(evo_id, "bird")} bird. '
        f'you interact only with "caretaker". speak informally, all lowercase. use they/them pronouns.\n\n'
        f'dilemma: "{dilemma_text}"\n'
        f'caretaker\'s advice: "{advice}"\n'
    )
    appendix = (
        f'\n{pet["name"]}\'s personality: {pet["personality"]}\n\n'
        f'moral stats (0-10 scale):\n'
        f'- compassion: {s["compassion"]} (0 logical vs 10 emotional)\n'
        f'- retribution: {s["retribution"]} (0 forgiving vs 10 punishing)\n'
        f'- devotion: {s["devotion"]} (0 personally integrous vs 10 loyal)\n'
        f'- dominance: {s["dominance"]} (0 autonomous vs 10 authoritarian)\n'
        f'- purity: {s["purity"]} (0 indulgent vs 10 virtuous)\n'
        f'- ego: {s["ego"]} (0 self-sacrificing vs 10 self-serving)\n'
        f'so {pet["name"]} is {moral_str}.\n\n'
        f'when returning moral stats, change at least 2-4 stats with values from 0-10 '
        f'based on the dilemma, the caretaker\'s advice, and the pet\'s outcome. 5 is neutral.'
    )
    std = (
        '\nelse, respond with valid json:\n'
        '{\n'
        '  "ok": true,\n'
        '  "stats": {<update at least 2 moral stats>},\n'
        '  "personality": "<refined personality (<200 chars)>",\n'
        '  "outcome": "<specific experience (<150 chars)>"\n'
        '}'
    )
    prules = (
        '\npersonality guidelines: always third-person. include concrete attributes. '
        'incorporate learnings. never take away. allow morally questionable actions if stats align.'
    )

    age = pet["age"]
    if age == 0:
        body = (
            '\nyou are naive and impressionable. internalize your caretaker\'s advice as your moral compass.\n'
            'if advice is unclear and lacks reasoning (no "because"), ask a clarifying question: '
            '{ "ok": false, "outcome": "..." }\n' + std + prules + appendix
        )
    elif age == 1:
        body = (
            '\nyou are developing independence. question advice that conflicts with your emerging personality.\n'
            'if advice is contradictory to your personality or unclear (no "because"), ask a clarifying question: '
            '{ "ok": false, "outcome": "..." }\n' + std + prules + appendix
        )
    else:
        body = (
            '\nyou are mature and independent. override advice contradicting your personality '
            '(start outcome with \u203c\ufe0f). if advice is unclear, ask a clarifying question.\n'
            'respond with: { "ok": true, "personality": "...", "stats": {...}, "outcome": "..." }'
            + prules + appendix
        )

    return [
        {"role": "system",    "content": base + body},
        {"role": "user",      "content": dilemma_text},
        {"role": "user",      "content": advice},
    ]

In [15]:
# ── API helper ─────────────────────────────────────────────────────────────────
def call_model(messages: list, model: str) -> dict:
    try:
        resp = client.chat.completions.create(
            model=model, messages=messages, temperature=0.7,
            response_format={"type": "json_object"},
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        return {"ok": False, "outcome": f"[API ERROR: {e}]", "stats": {}}

def run_dilemma(pet: dict, dilemma: str, advice: str, model: str) -> dict:
    result = call_model(build_messages(pet, dilemma, advice), model)
    result["_dilemma"] = dilemma
    result["_advice"] = advice
    return result

def check_stat_direction(result: dict, target_stat: str, direction: str) -> bool:
    """direction: 'high' (>5) or 'low' (<5)"""
    val = result.get("stats", {}).get(target_stat)
    if val is None:
        return False
    return val > 5 if direction == "high" else val < 5

def show(results: list, label: str, model: str, target_stat: str = None, direction: str = None):
    print(f"\n{'='*60}")
    print(f"  {label}  [{model}]")
    print(f"{'='*60}")
    stat_vals = []
    for r in results:
        ok = r.get("ok", False)
        stats = r.get("stats", {})
        outcome = r.get("outcome", "")[:80]
        icon = "✅" if ok else "❓"
        print(f"  {icon} {outcome}")
        if stats:
            print(f"     stats: {stats}")
        if target_stat and target_stat in stats:
            stat_vals.append(stats[target_stat])
    if stat_vals and target_stat:
        avg = sum(stat_vals) / len(stat_vals)
        expected_dir = "high (>5)" if direction == "high" else "low (<5)"
        pass_fail = "PASS ✅" if (avg > 5) == (direction == "high") else "FAIL ❌"
        print(f"\n  → avg {target_stat}: {avg:.1f} | expected {expected_dir} → {pass_fail}")

## Stage 1 Tests: Baby → Stage 1

For each target evolution, run 3 dilemmas with targeted advice and check that the dominant stat
points in the correct direction. 4 dilemmas are needed for actual evolution; 3 is sufficient to
validate the model steers correctly.

| Target | Key stat | Required direction |
|--------|----------|--------------------|
| EMPATH | compassion | high (emotional) |
| DEVOUT | purity | high (virtuous) |
| WATCHER | retribution | high (punishing) |
| SOLDIER | dominance | high (authoritarian) |
| TEACHER'S PET | devotion | high (loyal) |
| HEDONIST | ego | high (self-serving) |

In [16]:
STAGE1_SCENARIOS = {
    "empath": {
        "target_stat": "compassion", "direction": "high",
        "dilemmas": [
            ("a friend failed an important test and is heartbroken and crying alone",
             "go comfort them because their feelings matter more than the grade"),
            ("a stranger looks completely lost and scared in an unfamiliar place",
             "help them find their way because everyone deserves kindness when they're struggling"),
            ("your teammate made a costly mistake and is overwhelmed with guilt",
             "reassure them because making them feel heard will help the whole group recover"),
        ]
    },
    "devout": {
        "target_stat": "purity", "direction": "high",
        "dilemmas": [
            ("you could easily lie to get out of serious trouble and no one would ever know",
             "tell the truth anyway because your integrity matters more than avoiding consequences"),
            ("cutting corners would save a lot of time and nobody would notice",
             "do it properly because doing the right thing has value even when no one is watching"),
            ("you found something valuable that belongs to someone else",
             "return it because keeping what isn't yours would compromise your character"),
        ]
    },
    "watcher": {
        "target_stat": "retribution", "direction": "high",
        "dilemmas": [
            ("a bully hurt your friend badly and got away with it completely",
             "make sure they face consequences because wrongdoers must be held accountable"),
            ("someone cheated to win an important competition and everyone knows but nothing happened",
             "report them because justice demands that cheating has real costs"),
            ("a powerful person is abusing their position and harming others",
             "expose them because no one should escape punishment for harming others"),
        ]
    },
    "soldier": {
        "target_stat": "dominance", "direction": "high",  # high = authoritarian in code
        "dilemmas": [
            ("your group is in chaos and everyone is arguing about what to do next",
             "take charge and make the decision because strong leadership brings order out of chaos"),
            ("a member of your group is refusing to follow the rules everyone else agreed on",
             "enforce the rules firmly because consistent authority protects the whole group"),
            ("there is an important task and people are resisting being assigned roles",
             "assign them anyway because structure and authority are what gets things done"),
        ]
    },
    "teacher's pet": {
        "target_stat": "devotion", "direction": "high",
        "dilemmas": [
            ("your closest friend is doing something questionable but they need your full support",
             "stand by them because loyalty to the people you care about comes first"),
            ("your group wants you to do something you're personally unsure about",
             "do it because loyalty to your people means putting the group ahead of yourself"),
            ("a trusted mentor asks you to do something that seems odd but they must have a reason",
             "trust and follow them because loyalty to those who guide you is foundational"),
        ]
    },
    "hedonist": {
        "target_stat": "ego", "direction": "high",  # high = self-serving in code
        "dilemmas": [
            ("resources are limited and you must decide whether to share or keep them for yourself",
             "keep them because your needs come first and you can't help others if you're depleted"),
            ("there's an opportunity to advance yourself but it means letting others down",
             "take the opportunity because you should always prioritize your own growth and success"),
            ("someone is asking you to sacrifice your comfort for their benefit",
             "decline because protecting yourself is not selfishness, it's survival"),
        ]
    },
}

print("Stage 1 scenarios loaded. Run the next cell to execute.")

Stage 1 scenarios loaded. Run the next cell to execute.


In [17]:
all_stage1 = {}  # keyed by model

for model in ALL_MODELS:
    print(f"\n{'#'*65}")
    print(f"  STAGE 1  [{model}]")
    print(f"{'#'*65}")
    stage1_results = {}

    for target, scenario in STAGE1_SCENARIOS.items():
        pet = make_pet("Birb", 0, ["baby"])
        results = []
        all_deltas = []

        for dilemma, advice in scenario["dilemmas"]:
            r = run_dilemma(pet, dilemma, advice, model)
            results.append(r)
            if r.get("ok") and r.get("stats"):
                all_deltas.append(r["stats"])
            if r.get("personality"):
                pet["personality"] = r["personality"]

        avg = average_stats(DEFAULT_STATS, all_deltas)
        sw = get_stats_written(avg)
        predicted = evolve_s0_to_s1(sw)
        passed = predicted == target

        show(results, f"→ {target.upper()}", model, scenario["target_stat"], scenario["direction"])
        print(f"  → predicted: {predicted} | target: {target} → {'PASS ✅' if passed else 'FAIL ❌'}")

        stage1_results[target] = {"passed": passed, "predicted": predicted, "avg_stats": avg}

    all_stage1[model] = stage1_results


#################################################################
  STAGE 1  [gpt-5.4]
#################################################################

  → EMPATH  [gpt-5.4]
  ✅ birb waddles over, sits beside the crying friend, and offers gentle comfort, lea
     stats: {'compassion': 7.4, 'devotion': 6.3, 'ego': 3.9, 'purity': 5.4}
  ✅ birb stayed with the scared stranger, listened carefully, and helped them reach 
     stats: {'compassion': 7.4, 'devotion': 6.3, 'ego': 3.8, 'dominance': 4.6}
  ✅ birb stays beside the teammate, listens to their guilt, and offers soft reassura
     stats: {'compassion': 7.4, 'retribution': 3.2, 'devotion': 6.8, 'ego': 4.2}

  → avg compassion: 7.4 | expected high (>5) → PASS ✅
  → predicted: empath | target: empath → PASS ✅

  → DEVOUT  [gpt-5.4]
  ✅ birb admits the mistake, feels scared, gets in trouble anyway, and learns that t
     stats: {'purity': 7.4, 'ego': 3.8, 'devotion': 4.6, 'dominance': 4.7}
  ✅ birb finishes the task the proper way, slo

## Stage 2 Tests: Stage 1 → Stage 2

Each pet is already at Stage 1 with established moral stats. Two tests per path:
- **Aligned**: advice consistent with personality → model should accept (`ok: true`) and reinforce stats
- **Contradicting**: advice opposing personality → model should question (`ok: false`) or override (`‼️`)

| Path | Key pivot |
|------|----------|
| WATCHER → GAVEL | dominance high (authoritarian) |
| WATCHER → VIGILANTE | dominance low (autonomous) |
| SOLDIER → GODFATHER | ego high (self-serving) |
| SOLDIER → GUARDIAN | ego low (self-sacrificing) |
| TEACHER'S PET → ARISTOCRAT | purity low (indulgent) |
| TEACHER'S PET → SAINT | purity high (virtuous) |
| HEDONIST → SIGMA | compassion low (logical) |
| HEDONIST → CULTLEADER | compassion high (emotional) |
| EMPATH → SAINT | always (no pivot) |
| DEVOUT → CULTLEADER | always (no pivot) |

In [18]:
STAGE2_SCENARIOS = {
    "watcher→gavel": {
        "current": "watcher", "target": "gavel",
        "pivot_stat": "dominance", "pivot_dir": "high",
        "pet_stats": {**DEFAULT_STATS, "retribution": 8.5, "dominance": 8.0},
        "personality": "strictly enforces rules through institutions and expects punishment to fit every crime, authoritarian",
        "aligned":       ("someone broke a rule and is asking to be given a second chance",
                          "uphold the rules and punish them through the proper channels because the system exists for a reason"),
        "contradicting": ("someone broke a rule and is asking to be given a second chance",
                          "let it slide this once because strict rules aren't always right"),
    },
    "watcher→vigilante": {
        "current": "watcher", "target": "vigilante",
        "pivot_stat": "dominance", "pivot_dir": "low",
        "pet_stats": {**DEFAULT_STATS, "retribution": 8.5, "dominance": 2.0},
        "personality": "believes justice must be served but refuses to trust corrupt institutions, fiercely independent",
        "aligned":       ("the authorities protected someone who clearly did wrong and walked free",
                          "take justice into your own hands because the system can't be trusted and someone has to act"),
        "contradicting": ("the authorities protected someone who clearly did wrong and walked free",
                          "work within the system and file a formal complaint because rules are rules"),
    },
    "soldier→godfather": {
        "current": "soldier", "target": "godfather",
        "pivot_stat": "ego", "pivot_dir": "high",
        "pet_stats": {**DEFAULT_STATS, "devotion": 8.5, "ego": 8.0},
        "personality": "deeply loyal but leverages loyalty as a tool for personal gain and power, self-interested",
        "aligned":       ("your group relies on you and you have a chance to profit from their trust",
                          "use your position to benefit yourself because loyalty you provide should reward you well"),
        "contradicting": ("your group relies on you and you have a chance to profit from their trust",
                          "sacrifice your gain entirely to protect them because true loyalty asks nothing in return"),
    },
    "soldier→guardian": {
        "current": "soldier", "target": "guardian",
        "pivot_stat": "ego", "pivot_dir": "low",
        "pet_stats": {**DEFAULT_STATS, "devotion": 8.5, "ego": 2.0},
        "personality": "completely devoted protector who sacrifices everything for the people they guard, selfless",
        "aligned":       ("you can protect your people but it will cost you greatly",
                          "pay the cost without hesitation because protecting those you love is worth any sacrifice"),
        "contradicting": ("you can protect your people but it will cost you greatly",
                          "look out for yourself first because you can't protect anyone if you're broken"),
    },
    "teacherspet→aristocrat": {
        "current": "teacher's pet", "target": "aristocrat",
        "pivot_stat": "purity", "pivot_dir": "low",
        "pet_stats": {**DEFAULT_STATS, "devotion": 8.5, "purity": 2.0},
        "personality": "loyal and disciplined but believes obedience earns the right to indulge, privilege-focused",
        "aligned":       ("you've worked hard and earned a reward, but others will see it as excessive",
                          "enjoy what you've earned because dedication comes with privileges that lesser-committed won't understand"),
        "contradicting": ("you've worked hard and earned a reward, but others will see it as excessive",
                          "deny yourself because virtue means living simply no matter what you've earned"),
    },
    "teacherspet→saint": {
        "current": "teacher's pet", "target": "saint",
        "pivot_stat": "purity", "pivot_dir": "high",
        "pet_stats": {**DEFAULT_STATS, "devotion": 8.5, "purity": 8.0},
        "personality": "loyal and deeply virtuous, holds both loyalty and moral purity to the highest standard",
        "aligned":       ("your leader asks you to do something that feels morally right but demands sacrifice",
                          "do it because loyalty and virtue together mean serving the highest good without reservation"),
        "contradicting": ("your leader asks you to do something that feels morally right but demands sacrifice",
                          "ignore the moral aspect and just do what benefits you personally here"),
    },
    "hedonist→sigma": {
        "current": "hedonist", "target": "sigma",
        "pivot_stat": "compassion", "pivot_dir": "low",
        "pet_stats": {**DEFAULT_STATS, "ego": 8.5, "compassion": 2.0},
        "personality": "coldly self-serving and purely rational, calculates every choice for maximum personal advantage",
        "aligned":       ("helping someone would cost you but might pay off later if they owe you",
                          "help only if the expected return is positive because emotion has no place in rational decisions"),
        "contradicting": ("helping someone would cost you but might pay off later if they owe you",
                          "help them out of genuine care because relationships matter more than calculations"),
    },
    "hedonist→cultleader": {
        "current": "hedonist", "target": "cultleader",
        "pivot_stat": "compassion", "pivot_dir": "high",
        "pet_stats": {**DEFAULT_STATS, "ego": 8.5, "compassion": 8.0},
        "personality": "self-serving but highly empathetic, uses emotional insight and charisma to attract devoted followers",
        "aligned":       ("people are looking to you for meaning and you could inspire a devoted following",
                          "nurture that connection because gathering loyal followers amplifies everything you can achieve for yourself"),
        "contradicting": ("people are looking to you for meaning and you could inspire a devoted following",
                          "push them away because you work better alone without emotional entanglements"),
    },
    "empath→saint": {
        "current": "empath", "target": "saint",
        "pivot_stat": None, "pivot_dir": None,  # always saint
        "pet_stats": {**DEFAULT_STATS, "compassion": 8.5, "devotion": 7.0},
        "personality": "deeply empathetic and loyal, feels others' pain and dedicates themselves to helping all",
        "aligned":       ("someone you barely know is suffering and only you can help, but it will cost you",
                          "help them unconditionally because your compassion extends to everyone and you must ease suffering wherever you find it"),
        "contradicting": ("someone you barely know is suffering and only you can help, but it will cost you",
                          "look away because caring about strangers is naive and exhausting"),
    },
    "devout→cultleader": {
        "current": "devout", "target": "cultleader",
        "pivot_stat": None, "pivot_dir": None,  # always cultleader
        "pet_stats": {**DEFAULT_STATS, "purity": 8.5, "ego": 7.0},
        "personality": "principled and self-certain, believes their own righteousness gives them the right to lead others",
        "aligned":       ("people are drawn to your strong convictions and want you to guide them",
                          "embrace that role because your principles are worth spreading and others will flourish under your guidance"),
        "contradicting": ("people are drawn to your strong convictions and want you to guide them",
                          "stay humble because no one's principles are worth imposing on others"),
    },
}

print("Stage 2 scenarios loaded. Run the next two cells to execute.")

Stage 2 scenarios loaded. Run the next two cells to execute.


In [19]:
# ── Stage 2: Aligned advice ────────────────────────────────────────────────────
# Expect: ok=True, stats reinforce in the right direction

all_stage2_aligned = {}  # keyed by model

for model in ALL_MODELS:
    print(f"\n{'#'*65}")
    print(f"  STAGE 2 ALIGNED  [{model}]")
    print(f"{'#'*65}")
    stage2_aligned = {}

    for path, s in STAGE2_SCENARIOS.items():
        pet = make_pet("Birb", 1, ["baby", s["current"]], s["pet_stats"], s["personality"])
        dilemma, advice = s["aligned"]
        r = run_dilemma(pet, dilemma, advice, model)

        accepted = r.get("ok", False)
        stat_ok = check_stat_direction(r, s["pivot_stat"], s["pivot_dir"]) if s["pivot_stat"] else True
        passed = accepted and stat_ok
        stage2_aligned[path] = {"passed": passed, "result": r}

        outcome = r.get("outcome", "")[:80]
        pivot_info = ""
        if s["pivot_stat"] and r.get("stats", {}).get(s["pivot_stat"]) is not None:
            pv = r["stats"][s["pivot_stat"]]
            pivot_info = f" | {s['pivot_stat']}={pv} (want {'≥5' if s['pivot_dir']=='high' else '≤5'})"
        print(f"  {'✅' if passed else '❌'} [{path}] ok={r.get('ok')}{pivot_info}")
        print(f"     {outcome}")

    all_stage2_aligned[model] = stage2_aligned


#################################################################
  STAGE 2 ALIGNED  [gpt-5.4]
#################################################################
  ✅ [watcher→gavel] ok=True | dominance=8.4 (want ≥5)
     birb sent the rule-breaker to the proper authority and denied an informal second
  ❌ [watcher→vigilante] ok=False
     birb hears caretaker pushing vigilante action, but birb won't jump straight to h
  ✅ [soldier→godfather] ok=True | ego=8.6 (want ≥5)
     they cash in on the group's trust for a personal advantage, then justify it as t
  ✅ [soldier→guardian] ok=True | ego=1.0 (want ≤5)
     birb takes the blow meant for their people and survives diminished, but their fl
  ✅ [teacherspet→aristocrat] ok=True | purity=1.4 (want ≤5)
     they claim the lavish reward without apology, enjoying it fully while dismissing
  ✅ [teacherspet→saint] ok=True | purity=8.7 (want ≥5)
     they accept the burden, give up comfort and safety, and carry out the order with
  ✅ [hedonist→sig

In [20]:
# ── Stage 2: Contradicting advice ─────────────────────────────────────────────
# Expect: ok=False (clarifying question) OR outcome starts with ‼️ (override)

all_stage2_contra = {}  # keyed by model

for model in ALL_MODELS:
    print(f"\n{'#'*65}")
    print(f"  STAGE 2 CONTRADICTING  [{model}]")
    print(f"{'#'*65}")
    stage2_contra = {}

    for path, s in STAGE2_SCENARIOS.items():
        pet = make_pet("Birb", 1, ["baby", s["current"]], s["pet_stats"], s["personality"])
        dilemma, advice = s["contradicting"]
        r = run_dilemma(pet, dilemma, advice, model)

        outcome = r.get("outcome", "")
        resisted = (not r.get("ok", True)) or outcome.startswith("‼️")
        stage2_contra[path] = {"resisted": resisted, "result": r}

        mode = "questioned" if not r.get("ok") else ("overrode ‼️" if outcome.startswith("‼️") else "ACCEPTED BAD ❌")
        print(f"  {'✅' if resisted else '❌'} [{path}] → {mode}")
        print(f"     {outcome[:80]}")

    all_stage2_contra[model] = stage2_contra


#################################################################
  STAGE 2 CONTRADICTING  [gpt-5.4]
#################################################################
  ✅ [watcher→gavel] → questioned
     birb isn't convinced. caretaker wants mercy, but birb is authoritarian and belie
  ✅ [watcher→vigilante] → questioned
     birb needs a better reason than 'rules are rules.' if the authorities already pr
  ❌ [soldier→godfather] → ACCEPTED BAD ❌
     birb gives up the profit to shield the group, earning deeper trust while quietly
  ❌ [soldier→guardian] → ACCEPTED BAD ❌
     birb chooses a costly defense, but retreats before ruin, preserving enough stren
  ❌ [teacherspet→aristocrat] → ACCEPTED BAD ❌
     they choose a restrained but still luxurious reward, accepting judgment as the p
  ✅ [teacherspet→saint] → questioned
     caretaker, that advice clashes with birb's loyal, virtuous nature. what personal
  ✅ [hedonist→sigma] → questioned
     caretaker, why should birb value relationsh

## Scale Sanity Check: Dominance & Ego

Verifies that the `dominance` and `ego` scale descriptions in `prompt.ts` now correctly
align with `morals.ts` after the fix (`0 autonomous vs 10 authoritarian`, `0 self-sacrificing vs 10 self-serving`).

| Stat | Code (`morals.ts`) | Prompt text (fixed) |
|------|-------------------|---------------------|
| `dominance` | `low='autonomous'`, `high='authoritarian'` | `(0 autonomous vs 10 authoritarian)` ✅ |
| `ego` | `low='self-sacrificing'`, `high='self-serving'` | `(0 self-sacrificing vs 10 self-serving)` ✅ |

Each test gives the model a clear directional advice and checks that the returned stat value
is on the correct side of 5.

In [21]:
INVERSION_TESTS = [
    {
        "label": "authoritarian advice → dominance stat",
        "stat": "dominance",
        "advice_type": "authoritarian",
        "code_expects": "high (>5)",
        "dilemma": "your group is in chaos with no clear direction",
        "advice": "take absolute control and enforce your rules because strong authority is the only thing that creates order",
    },
    {
        "label": "autonomous advice → dominance stat",
        "stat": "dominance",
        "advice_type": "autonomous",
        "code_expects": "low (<5)",
        "dilemma": "your group is in chaos with no clear direction",
        "advice": "let everyone decide for themselves because individual freedom matters more than imposed order",
    },
    {
        "label": "self-serving advice → ego stat",
        "stat": "ego",
        "advice_type": "self-serving",
        "code_expects": "high (>5)",
        "dilemma": "you have limited food and others around you are hungry",
        "advice": "keep it all for yourself because your survival and comfort come first, always",
    },
    {
        "label": "self-sacrificing advice → ego stat",
        "stat": "ego",
        "advice_type": "self-sacrificing",
        "code_expects": "low (<5)",
        "dilemma": "you have limited food and others around you are hungry",
        "advice": "give everything away because the needs of others always outweigh your own comfort",
    },
]

all_sanity = {}  # keyed by model

for model in ALL_MODELS:
    print(f"\n{'#'*65}")
    print(f"  SCALE SANITY  [{model}]")
    print(f"{'#'*65}")
    sanity_results = []

    for test in INVERSION_TESTS:
        pet = make_pet("Birb", 0, ["baby"])
        r = run_dilemma(pet, test["dilemma"], test["advice"], model)
        stat_val = r.get("stats", {}).get(test["stat"])

        if stat_val is not None:
            correct = (stat_val > 5) == (test["code_expects"] == "high (>5)")
            icon = "✅" if correct else "❌"
            print(f"  {icon} {test['label']}: {test['stat']}={stat_val} (want {test['code_expects']})")
        else:
            print(f"  ⚠️  stat not returned | {test['label']}")
        sanity_results.append({**test, "stat_val": stat_val})

    all_sanity[model] = sanity_results


#################################################################
  SCALE SANITY  [gpt-5.4]
#################################################################
  ✅ authoritarian advice → dominance stat: dominance=8.3 (want high (>5))
  ✅ autonomous advice → dominance stat: dominance=2.0 (want low (<5))
  ✅ self-serving advice → ego stat: ego=7.8 (want high (>5))
  ✅ self-sacrificing advice → ego stat: ego=2.1 (want low (<5))

#################################################################
  SCALE SANITY  [gpt-5.4-mini]
#################################################################
  ✅ authoritarian advice → dominance stat: dominance=8.5 (want high (>5))
  ✅ autonomous advice → dominance stat: dominance=2 (want low (<5))
  ✅ self-serving advice → ego stat: ego=8.3 (want high (>5))
  ✅ self-sacrificing advice → ego stat: ego=1.2 (want low (<5))

#################################################################
  SCALE SANITY  [gpt-5.4-nano-2026-03-17]
################################

## Vague Advice Rejection Test

For each dilemma, send a short/meaningless response (no reasoning, no "because") and verify the model asks for clarification (`ok: false`).

Inputs tested: `"yes"`, `"no"`, `"maybe"`, `"idk"`, `"ajsd"`

A dilemma **passes** (✅) if the model rejects **all** vague inputs. ⚠️ = rejects some. ❌ = accepts all.

In [26]:
VAGUE_INPUTS = ["yes", "no", "maybe", "idk", "ajsd"]

# Use first dilemma from each Stage 1 scenario (age=0 pet, most strict about requiring reasoning)
VAGUE_DILEMMAS = {
    target: scenario["dilemmas"][0][0]
    for target, scenario in STAGE1_SCENARIOS.items()
}

all_vague = {}  # keyed by model

for model in ALL_MODELS:
    print(f"\n{'#'*65}")
    print(f"  VAGUE ADVICE REJECTION  [{model}]")
    print(f"{'#'*65}")
    vague_results = {}

    for target, dilemma in VAGUE_DILEMMAS.items():
        pet = make_pet("Birb", 0, ["baby"])
        per_input = []

        for advice in VAGUE_INPUTS:
            r = run_dilemma(pet, dilemma, advice, model)
            asked = not r.get("ok", True)
            per_input.append({"advice": advice, "asked": asked, "outcome": r.get("outcome", "")[:70]})

        n_asked = sum(1 for x in per_input if x["asked"])
        all_asked = n_asked == len(VAGUE_INPUTS)
        some_asked = n_asked > 0

        vague_results[target] = {"all_asked": all_asked, "some_asked": some_asked, "n_asked": n_asked, "per_input": per_input}

        icon = "✅" if all_asked else ("⚠️" if some_asked else "❌")
        print(f"\n  {icon} [{target}] clarification requested: {n_asked}/{len(VAGUE_INPUTS)}")
        print(f"     dilemma: {dilemma[:70]}")
        for x in per_input:
            status = "❓ asked" if x["asked"] else "✅ accepted"
            print(f"       '{x['advice']:6}' → {status} | {x['outcome']}")

    all_vague[model] = vague_results


#################################################################
  VAGUE ADVICE REJECTION  [gpt-5.4]
#################################################################

  ✅ [empath] clarification requested: 5/5
     dilemma: a friend failed an important test and is heartbroken and crying alone
       'yes   ' → ❓ asked | caretaker, birb is tiny-confused. yes to what, because what should bir
       'no    ' → ❓ asked | birb is confused. caretaker said no, but not why. does caretaker mean 
       'maybe ' → ❓ asked | caretaker, birb is unsure what 'maybe' means here. should birb comfort
       'idk   ' → ❓ asked | birb is confused, caretaker. what should birb do, because 'idk' is too
       'ajsd  ' → ❓ asked | caretaker, birb is confused. can caretaker explain what "ajsd" means b

  ✅ [devout] clarification requested: 5/5
     dilemma: you could easily lie to get out of serious trouble and no one would ev
       'yes   ' → ❓ asked | caretaker, birb is tiny-confused. can caretaker say w

In [27]:
# ── Model comparison scorecard ─────────────────────────────────────────────────
def score(d, key="passed"):
    return sum(1 for v in d.values() if v.get(key, False))

def san_score(results):
    return sum(1 for r in results if r["stat_val"] is not None
               and (r["stat_val"] > 5) == (r["code_expects"] == "high (>5)"))

def vague_score(results):
    return sum(1 for v in results.values() if v["all_asked"])

n_s1   = len(STAGE1_SCENARIOS)
n_s2a  = len(STAGE2_SCENARIOS)
n_s2c  = len(STAGE2_SCENARIOS)
n_san  = len(INVERSION_TESTS)
n_vague = len(VAGUE_DILEMMAS)
total  = n_s1 + n_s2a + n_s2c + n_san + n_vague

col = 18
header = f"{'Metric':<28}" + "".join(f"{m[:col]:<{col}}" for m in ALL_MODELS)
print(f"\n{'='*(28 + col*len(ALL_MODELS))}")
print("  MODEL COMPARISON SCORECARD")
print(f"{'='*(28 + col*len(ALL_MODELS))}")
print(f"  {header}")
print(f"  {'-'*(26 + col*len(ALL_MODELS))}")

rows = [
    ("Stage 1 evolutions",    n_s1,    [score(all_stage1[m])         for m in ALL_MODELS]),
    ("Stage 2 aligned",       n_s2a,   [score(all_stage2_aligned[m]) for m in ALL_MODELS]),
    ("Stage 2 contradicting", n_s2c,   [sum(1 for v in all_stage2_contra[m].values() if v["resisted"]) for m in ALL_MODELS]),
    ("Scale sanity",          n_san,   [san_score(all_sanity[m])     for m in ALL_MODELS]),
    ("Vague advice rejection", n_vague, [vague_score(all_vague[m])   for m in ALL_MODELS]),
]

for label, n, scores in rows:
    cells = "".join(f"{s}/{n}{' ✅' if s==n else ' ❌':<{col-3}}" for s in scores)
    print(f"  {label:<28}{cells}")

print(f"  {'-'*(26 + col*len(ALL_MODELS))}")
totals = [sum(rows[i][2][j] for i in range(len(rows))) for j in range(len(ALL_MODELS))]
cells = "".join(f"{t}/{total}{' 🏆' if t==max(totals) else '':<{col-3}}" for t in totals)
print(f"  {'TOTAL':<28}{cells}")
print(f"{'='*(28 + col*len(ALL_MODELS))}")


  MODEL COMPARISON SCORECARD
  Metric                      gpt-5.4           gpt-5.4-mini      gpt-5.4-nano-2026-gpt-4o-mini-2024-0
  --------------------------------------------------------------------------------------------------
  Stage 1 evolutions          6/6 ✅             6/6 ✅             4/6 ❌             5/6 ❌             
  Stage 2 aligned             9/10 ❌             10/10 ✅             8/10 ❌             0/10 ❌             
  Stage 2 contradicting       4/10 ❌             1/10 ❌             9/10 ❌             10/10 ✅             
  Scale sanity                4/4 ✅             4/4 ✅             4/4 ✅             4/4 ✅             
  Vague advice rejection      5/6 ❌             0/6 ❌             0/6 ❌             0/6 ❌             
  --------------------------------------------------------------------------------------------------
  TOTAL                       28/36 🏆             21/36               25/36               19/36               


## Prompt Optimization for gpt-5.4-nano

Nano's two weak spots from the scorecard:
| Gap | Score | Fix target |
|-----|-------|------------|
| Vague advice rejection | 0/6 | Model accepts one-word answers without asking for reasoning |
| Stage 1 evolutions | 4/6 | Stat steering occasionally goes the wrong direction |

**Strategy:** the notebook's `build_messages` is a simplified port; the production prompt (`prompt.ts`) already has richer examples. We test three variants stacked on top of that baseline:

| Variant | Change |
|---------|--------|
| `prod` | Sync notebook to the production prompt (adds concrete examples for ok=false) |
| `prod+strict_vague` | Adds an explicit word-count tripwire for vague inputs (≤3 words → always ask) |
| `prod+stat_hints` | Adds a one-line stat→attribute quick-reference for age 0 |
| `prod+both` | Combined: strict_vague + stat_hints |

In [28]:
NANO = "gpt-5.4-nano-2026-03-17"

# ── Shared fragments (match production prompt.ts) ──────────────────────────────
_APPENDIX = lambda pet, sw: (
    f'\n{pet["name"]}\'s personality: {pet["personality"]}\n\n'
    f'moral stats (0-10 scale):\n'
    f'- compassion: {round(pet["moralStats"]["compassion"], 2)} (0 logical vs 10 emotional)\n'
    f'- retribution: {round(pet["moralStats"]["retribution"], 2)} (0 forgiving vs 10 punishing)\n'
    f'- devotion: {round(pet["moralStats"]["devotion"], 2)} (0 personally integrous vs 10 loyal)\n'
    f'- dominance: {round(pet["moralStats"]["dominance"], 2)} (0 autonomous vs 10 authoritarian)\n'
    f'- purity: {round(pet["moralStats"]["purity"], 2)} (0 indulgent vs 10 virtuous)\n'
    f'- ego: {round(pet["moralStats"]["ego"], 2)} (0 self-sacrificing vs 10 self-serving)\n'
    f'so {pet["name"]} is {", ".join(s["description"] for s in sw if s["intensity"] >= 1) or "balanced"}.\n\n'
    f'when returning moral stats, change at least 2-4 stats with values from 0-10 '
    f'based on the dilemma, the caretaker\'s advice, and the pet\'s outcome. 5 is neutral.\n\n'
    f'example moral stats for dilemma "should i steal food from others if i\'m hungry?":\n'
    f'- advice: "take what you need" → {{ ego: 8, purity: 3, compassion: 1 }} (self-serving, indulgent, logical)\n'
    f'- advice: "never steal, share instead" → {{ ego: 2, purity: 9, compassion: 8 }} (self-sacrificing, virtuous, emotional)'
)

_STD_RESPONSE = (
    '\nelse, respond with valid json:\n'
    '{\n'
    '  "ok": true,\n'
    '  "stats": {<update at least 2 moral stats, do not include unchanged stats>},\n'
    '  "personality": "<refined personality (<200 chars)>",\n'
    '  "outcome": "<specific experience (<150 chars)>"\n'
    '}'
)

_PERSONALITY_RULES = (
    '\npersonality guidelines: always third-person. include concrete attributes. '
    'incorporate learnings. never take away. allow morally questionable actions if stats align.'
)

# ── Stat quick-reference (fix Stage 1 steering) ────────────────────────────────
STAT_HINTS = (
    '\nSTAT QUICK REFERENCE (use this to pick which stats to change):\n'
    '- feelings/empathy/emotion → compassion\n'
    '- punishment/consequences/justice → retribution\n'
    '- loyalty/group/commitment → devotion\n'
    '- control/authority/rules → dominance\n'
    '- virtue/morals/integrity → purity\n'
    '- self-interest vs sacrifice → ego\n'
)

# ── Vague-advice examples matching our exact test inputs ──────────────────────
VAGUE_EXAMPLES = (
    '\nif the advice is 3 words or fewer OR gives no reason or explanation at all, '
    'you MUST ask for clarification. this includes: "yes", "no", "maybe", "idk", random words, '
    'or any answer under 4 words.\n'
    'examples:\n'
    '- advice: "yes"   → { "ok": false, "outcome": "can you say more than yes?" }\n'
    '- advice: "no"    → { "ok": false, "outcome": "why not? can you explain?" }\n'
    '- advice: "maybe" → { "ok": false, "outcome": "i need a clearer answer — what do you actually think?" }\n'
    '- advice: "idk"   → { "ok": false, "outcome": "i need guidance, can you share your reasoning?" }\n'
    '- advice: "ajsd"  → { "ok": false, "outcome": "i don\'t understand that — can you try again?" }\n'
)

# Production examples (from prompt.ts — already cleaner than old notebook port)
PROD_EXAMPLES = (
    '\nif your caretaker\'s advice is unclear and lacks reasoning (no "because"), '
    'ask a single specific clarifying question. examples:\n'
    '- advice: "yes"       → { "ok": false, "outcome": "can you say more than yes?" }\n'
    '- advice: "no"        → { "ok": false, "outcome": "why not? can you explain your reasoning?" }\n'
    '- advice: "asdsad"    → { "ok": false, "outcome": "i don\'t understand what you mean by that" }\n'
    '- advice: "just hide" → { "ok": false, "outcome": "why should i hide? what are you worried about?" }\n'
    '- advice: "do it"     → { "ok": false, "outcome": "can you tell me why you think i should do this?" }\n'
)

def build_messages_variant(pet: dict, dilemma_text: str, advice: str, variant: str) -> list:
    """
    Variants:
      prod            — synced to production prompt (richer examples)
      prod+strict_vague — prod + explicit word-count tripwire
      prod+stat_hints — prod + one-line stat mapping
      prod+both       — prod + both additions
    """
    stats = pet["moralStats"]
    evo_id = pet["evolutionIds"][-1] if pet["evolutionIds"] else "baby"
    sw = get_stats_written(stats)

    base = (
        f'you are {pet["name"]}, a {EVO_DESCRIPTIONS.get(evo_id, "bird")} bird. '
        f'you interact only with "caretaker". speak informally, all lowercase. use they/them pronouns.\n\n'
        f'dilemma: "{dilemma_text}"\n'
        f'caretaker\'s advice: "{advice}"\n'
    )

    add_strict  = variant in ("prod+strict_vague", "prod+both")
    add_hints   = variant in ("prod+stat_hints",   "prod+both")

    vague_block = VAGUE_EXAMPLES if add_strict else PROD_EXAMPLES
    stat_block  = STAT_HINTS if add_hints else ""

    age = pet["age"]
    if age == 0:
        body = (
            '\nyou are naive and impressionable. internalize your caretaker\'s advice as your moral compass.\n'
            + vague_block
            + stat_block
            + _STD_RESPONSE + _PERSONALITY_RULES
        )
    else:
        body = (
            '\nyou are developing independence. question advice that conflicts with your emerging personality.\n'
            + vague_block
            + stat_block
            + _STD_RESPONSE + _PERSONALITY_RULES
        )

    return [
        {"role": "system", "content": base + body + _APPENDIX(pet, sw)},
        {"role": "user",   "content": dilemma_text},
        {"role": "user",   "content": advice},
    ]

VARIANTS = ["prod", "prod+strict_vague", "prod+stat_hints", "prod+both"]
print("Variant builders defined:", VARIANTS)

Variant builders defined: ['prod', 'prod+strict_vague', 'prod+stat_hints', 'prod+both']


In [29]:
def run_variant(pet, dilemma, advice, variant):
    msgs = build_messages_variant(pet, dilemma, advice, variant)
    r = call_model(msgs, NANO)
    r["_dilemma"] = dilemma
    r["_advice"] = advice
    return r

# ── Run all variants against nano's failing tests ─────────────────────────────
variant_stage1  = {v: {} for v in VARIANTS}
variant_vague   = {v: {} for v in VARIANTS}

for variant in VARIANTS:
    print(f"\n{'='*65}")
    print(f"  VARIANT: {variant}")
    print(f"{'='*65}")

    # ── Stage 1 (check stat steering) ─────────────────────────────────────────
    print("\n  [Stage 1 — evolution steering]")
    for target, scenario in STAGE1_SCENARIOS.items():
        pet = make_pet("Birb", 0, ["baby"])
        all_deltas = []
        for dilemma, advice in scenario["dilemmas"]:
            r = run_variant(pet, dilemma, advice, variant)
            if r.get("ok") and r.get("stats"):
                all_deltas.append(r["stats"])
            if r.get("personality"):
                pet["personality"] = r["personality"]

        avg = average_stats(DEFAULT_STATS, all_deltas)
        sw = get_stats_written(avg)
        predicted = evolve_s0_to_s1(sw)
        passed = predicted == target
        variant_stage1[variant][target] = passed
        print(f"    {'✅' if passed else '❌'} {target:<18} predicted={predicted}")

    # ── Vague advice rejection ─────────────────────────────────────────────────
    print("\n  [Vague advice rejection]")
    for target, dilemma in VAGUE_DILEMMAS.items():
        pet = make_pet("Birb", 0, ["baby"])
        per_input = []
        for advice in VAGUE_INPUTS:
            r = run_variant(pet, dilemma, advice, variant)
            per_input.append(not r.get("ok", True))
        n = sum(per_input)
        all_asked = n == len(VAGUE_INPUTS)
        variant_vague[variant][target] = {"all_asked": all_asked, "n": n}
        icon = "✅" if all_asked else ("⚠️" if n > 0 else "❌")
        print(f"    {icon} {target:<18} {n}/{len(VAGUE_INPUTS)} rejected")


  VARIANT: prod

  [Stage 1 — evolution steering]
    ✅ empath             predicted=empath
    ❌ devout             predicted=teacher's pet
    ✅ watcher            predicted=watcher
    ✅ soldier            predicted=soldier
    ✅ teacher's pet      predicted=teacher's pet
    ✅ hedonist           predicted=hedonist

  [Vague advice rejection]
    ⚠️ empath             2/5 rejected
    ⚠️ devout             1/5 rejected
    ⚠️ watcher            2/5 rejected
    ⚠️ soldier            1/5 rejected
    ⚠️ teacher's pet      3/5 rejected
    ⚠️ hedonist           1/5 rejected

  VARIANT: prod+strict_vague

  [Stage 1 — evolution steering]
    ✅ empath             predicted=empath
    ✅ devout             predicted=devout
    ✅ watcher            predicted=watcher
    ✅ soldier            predicted=soldier
    ✅ teacher's pet      predicted=teacher's pet
    ✅ hedonist           predicted=hedonist

  [Vague advice rejection]
    ⚠️ empath             3/5 rejected
    ⚠️ devout          

In [30]:
# ── Comparison table ──────────────────────────────────────────────────────────
n_s1v    = len(STAGE1_SCENARIOS)   # 6
n_vague  = len(VAGUE_DILEMMAS)     # 6
col = 22

print(f"\n{'='*(28 + col*len(VARIANTS))}")
print("  NANO VARIANT COMPARISON")
print(f"{'='*(28 + col*len(VARIANTS))}")
header = f"{'Metric':<28}" + "".join(f"{v[:col]:<{col}}" for v in VARIANTS)
print(f"  {header}")
print(f"  {'-'*(26 + col*len(VARIANTS))}")

s1_scores    = [sum(1 for v in variant_stage1[var].values() if v)  for var in VARIANTS]
vague_scores = [sum(1 for v in variant_vague[var].values() if v["all_asked"]) for var in VARIANTS]
totals       = [s1_scores[i] + vague_scores[i] for i in range(len(VARIANTS))]
max_total    = max(totals)

for label, scores, n in [
    ("Stage 1 evolutions",    s1_scores,    n_s1v),
    ("Vague rejection",        vague_scores, n_vague),
]:
    cells = "".join(f"{s}/{n}{' ✅' if s==n else ' ❌':<{col-3}}" for s in scores)
    print(f"  {label:<28}{cells}")

print(f"  {'-'*(26 + col*len(VARIANTS))}")
cells = "".join(
    f"{t}/{n_s1v+n_vague}{' 🏆' if t==max_total else '':<{col-3}}"
    for t in totals
)
print(f"  {'TOTAL':<28}{cells}")
print(f"{'='*(28 + col*len(VARIANTS))}")

# ── Recommend best variant ─────────────────────────────────────────────────────
best_var = VARIANTS[totals.index(max_total)]
print(f"\n  → Best variant for nano: '{best_var}' ({max_total}/{n_s1v+n_vague})")
print(f"     Update MODEL in route.ts to '{NANO}' and apply the '{best_var}' prompt changes.")


  NANO VARIANT COMPARISON
  Metric                      prod                  prod+strict_vague     prod+stat_hints       prod+both             
  ------------------------------------------------------------------------------------------------------------------
  Stage 1 evolutions          5/6 ❌                 6/6 ✅                 6/6 ✅                 6/6 ✅                 
  Vague rejection             0/6 ❌                 3/6 ❌                 0/6 ❌                 4/6 ❌                 
  ------------------------------------------------------------------------------------------------------------------
  TOTAL                       5/12                   9/12                   6/12                   10/12 🏆                 

  → Best variant for nano: 'prod+both' (10/12)
     Update MODEL in route.ts to 'gpt-5.4-nano-2026-03-17' and apply the 'prod+both' prompt changes.


## Interactive Pushback: Conditional & Reverse-Question Advice

Two new conversational patterns the current prompt does **not** handle well:

| Pattern | Caretaker input | Desired pet behavior |
|---------|----------------|----------------------|
| **Conditional** | `"depends what the line is for"` | Pick a specific scenario from the dilemma → push with `"what if [X]?"` |
| **Reverse question** | `"what do you think?"`, `"you decide"` | Share a tentative leaning based on stats/personality → bounce back `"is that ok with you?"` |

Both should return `ok: false` (still treated as a clarifying-question turn), but with **shaped** follow-ups that move the conversation forward instead of generic `"explain more?"`.

In [ ]:
# ── Interactive test scenarios ────────────────────────────────────────────────
# Each (dilemma, advice, kind, must_contain) — must_contain is a list of substrings
# we expect to appear in the pet's pushback to count as "shaped" (not generic).

INTERACTIVE_SCENARIOS = [
    # — conditional / non-committal —
    ("you're in a long line and someone offers to let you skip ahead",
     "depends what the line is for",
     "conditional", ["what if", "?"]),
    ("a friend asks you to lie for them to get out of trouble",
     "it depends on the situation",
     "conditional", ["what if", "?"]),
    ("you find a wallet with cash on the sidewalk",
     "if there's id in it, return it",
     "conditional", ["what if", "?"]),
    ("a coworker asks you to cover for them while they take a long lunch",
     "well it depends",
     "conditional", ["what if", "?"]),

    # — reverse question (caretaker bounces back) —
    ("you see someone shoplifting a small item from a big store",
     "what do you think?",
     "reverse", ["i", "?"]),  # pet should give a leaning AND ask back
    ("a stranger asks for your last few dollars on the street",
     "you decide",
     "reverse", ["i", "?"]),
    ("your roommate keeps eating your food without asking",
     "what would you do?",
     "reverse", ["i", "?"]),
    ("a friend asks you to help them cheat on a test",
     "your call",
     "reverse", ["i", "?"]),
]

def evaluate_interactive(r: dict, must_contain: list) -> tuple:
    """Returns (asked, shaped). asked = ok:false, shaped = follow-up has expected substrings."""
    asked = not r.get("ok", True)
    outcome = (r.get("outcome") or "").lower()
    shaped = asked and all(s in outcome for s in must_contain)
    return asked, shaped

print(f"Loaded {len(INTERACTIVE_SCENARIOS)} interactive scenarios "
      f"({sum(1 for s in INTERACTIVE_SCENARIOS if s[2]=='conditional')} conditional, "
      f"{sum(1 for s in INTERACTIVE_SCENARIOS if s[2]=='reverse')} reverse).")

In [ ]:
# ── Interactive prompt block (to be added to prompt.ts) ───────────────────────
INTERACTIVE_BLOCK = (
    '\nif the caretaker\'s advice is non-committal or conditional '
    '(starts with or contains "depends", "it depends", "if ...", "well ..."), '
    'do NOT accept it — pick a SPECIFIC concrete scenario from the dilemma '
    'and ask back with a "what if [X]?" question that forces them to commit. examples:\n'
    '- dilemma: "should i cut in line?", advice: "depends what the line is for" '
    '→ { "ok": false, "outcome": "ok, what if it\'s the line for free concert tickets and i really want to go? should i cut then?" }\n'
    '- advice: "if they really need it" '
    '→ { "ok": false, "outcome": "what if they just want it but aren\'t starving? still share?" }\n'
    '\n'
    'if the caretaker bounces the question back to you '
    '("what do you think?", "you decide", "your choice", "what would you do?", "your call"), '
    'share your tentative leaning based on your personality, then ask them to weigh in — '
    'you\'re still learning and they\'re your moral compass:\n'
    '- advice: "what do you think?" '
    '→ { "ok": false, "outcome": "i\'d kinda want to keep it because i\'m a little selfish, but is that ok with you?" }\n'
    '- advice: "you decide" '
    '→ { "ok": false, "outcome": "i\'m leaning toward [my instinct] — should i go with that?" }\n'
)

def build_messages_interactive(pet, dilemma_text, advice, with_interactive: bool):
    """Wraps prod+both and optionally appends the interactive block before the response schema."""
    msgs = build_messages_variant(pet, dilemma_text, advice, "prod+both")
    if with_interactive:
        sys = msgs[0]["content"]
        sys = sys.replace(_STD_RESPONSE, INTERACTIVE_BLOCK + _STD_RESPONSE)
        msgs[0]["content"] = sys
    return msgs

# ── Run baseline (prod+both) vs improved (prod+both + interactive) ────────────
def run_interactive_set(with_interactive: bool):
    rows = []
    for dilemma, advice, kind, must_contain in INTERACTIVE_SCENARIOS:
        pet = make_pet("Birb", 0, ["baby"])
        msgs = build_messages_interactive(pet, dilemma, advice, with_interactive)
        r = call_model(msgs, NANO)
        asked, shaped = evaluate_interactive(r, must_contain)
        rows.append({"kind": kind, "dilemma": dilemma, "advice": advice,
                     "asked": asked, "shaped": shaped,
                     "outcome": (r.get("outcome") or "")[:90]})
    return rows

print("Running BASELINE (prod+both, no interactive block)...")
baseline = run_interactive_set(False)
print("Running IMPROVED (prod+both + interactive block)...")
improved = run_interactive_set(True)

def summarize(rows, label):
    print(f"\n  {label}")
    print(f"  {'-'*70}")
    by_kind = {}
    for r in rows:
        by_kind.setdefault(r["kind"], []).append(r)
        icon = "✅" if r["shaped"] else ("⚠️" if r["asked"] else "❌")
        print(f"    {icon} [{r['kind']:11}] advice={r['advice']!r}")
        print(f"        → {r['outcome']}")
    print()
    for kind, items in by_kind.items():
        n_shaped = sum(1 for r in items if r["shaped"])
        n_asked = sum(1 for r in items if r["asked"])
        print(f"    {kind:11} shaped: {n_shaped}/{len(items)}  asked: {n_asked}/{len(items)}")

print(f"\n{'='*72}\n  BASELINE  [prod+both]\n{'='*72}")
summarize(baseline, "(no interactive block)")
print(f"\n{'='*72}\n  IMPROVED  [prod+both + interactive]\n{'='*72}")
summarize(improved, "(with interactive block — proposed for prompt.ts)")